In [10]:
# Load env variables and create client
import base64
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [11]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": getattr(message, "content", message),
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": getattr(message, "content", message),
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
    stream=False
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    if stream:
        message = client.messages.stream(**params)
    else:   
        message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [12]:
# Fire risk assessment prompt
prompt = """
Analyze the attached satellite image of a property with these specific steps:

1. Residence identification: Locate the primary residence on the property by looking for:
   - The largest roofed structure 
   - Typical residential features (driveway connection, regular geometry)
   - Distinction from other structures (garages, sheds, pools)
   Describe the residence's location relative to property boundaries and other features.

2. Tree overhang analysis: Examine all trees near the primary residence:
   - Identify any trees whose canopy extends directly over any portion of the roof
   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75-100%)
   - Note particularly dense areas of overhang

3. Fire risk assessment: For any overhanging trees, evaluate:
   - Potential wildfire vulnerability (ember catch points, continuous fuel paths to structure)
   - Proximity to chimneys, vents, or other roof openings if visible
   - Areas where branches create a "bridge" between wildland vegetation and the structure
   
4. Defensible space identification: Assess the property's overall vegetative structure:
   - Identify if trees connect to form a continuous canopy over or near the home
   - Note any obvious fuel ladders (vegetation that can carry fire from ground to tree to roof)

5. Fire risk rating: Based on your analysis, assign a Fire Risk Rating from 1-4:
   - Rating 1 (Low Risk): No tree branches overhanging the roof, good defensible space around the structure
   - Rating 2 (Moderate Risk): Minimal overhang (<25% of roof), some separation between tree canopies
   - Rating 3 (High Risk): Significant overhang (25-50% of roof), connected tree canopies, multiple points of vulnerability
   - Rating 4 (Severe Risk): Extensive overhang (>50% of roof), dense vegetation against structure, numerous ember catch points, limited defensible space

For each item above (1-5), write one sentence summarizing your findings, with your final response being the numeric Fire Risk Rating (1-4) with a brief justification.
"""

In [13]:
# # TODO: Read image data, feed into Claude
# with open("prop1.png", "rb") as f:
#     img_bytes = base64.b64encode(f.read()).decode("utf-8")

# messages = []


# add_user_message(messages, [
#     {
#         "type":"image",
#         "source": {
#             "type": "base64",
#             "media_type": "image/png",
#             "data": img_bytes
#         }
#     },
#     {
#         "type": "text",
#         "text": prompt
#     }
# ])

# chat_response = chat(messages, temperature=0.7)

# chat_response



In [14]:
# print(text_from_message(chat_response))

In [15]:
# TODO: Read image data, feed into Claude
with open("prop1.png", "rb") as f:
    img_bytes = base64.b64encode(f.read()).decode("utf-8")

messages = []


add_user_message(messages, [
    {
        "type":"image",
        "source": {
            "type": "base64",
            "media_type": "image/png",
            "data": img_bytes
        }
    },
    {
        "type": "text",
        "text": prompt
    }
])

chat_response = chat(messages, temperature=0.7, stream=True)

with chat_response as stream:
    for chunk in stream:
        if chunk.type == "text":
            print(chunk.text, end="", flush=True)



# Satellite Image Fire Risk Analysis

**1. Residence Identification:**
The primary residence is located in the center of the image, identifiable by its rectangular roofed structure with a light gray/tan colored roof, positioned within a cleared area surrounded by dense vegetation on all sides.

**2. Tree Overhang Analysis:**
Multiple trees have canopies that extend directly over portions of the residence roof, with overhang coverage estimated at 50-75% of the total roof area, particularly concentrated on the northern and eastern sections of the structure where dark tree canopies are clearly visible overlapping the roof surface.

**3. Fire Risk Assessment:**
The overhanging trees create significant wildfire vulnerability by providing direct pathways for embers to reach the roof, with dense canopy coverage creating multiple ember catch points and establishing continuous fuel connections between surrounding forest vegetation and the structure itself.

**4. Defensible Space Identification:

In [16]:
# # Example: use extended thinking with the same image prompt

# with open("prop1.png", "rb") as f:
#     img_bytes = base64.b64encode(f.read()).decode("utf-8")

# messages = []

# add_user_message(messages, [
#     {
#         "type": "image",
#         "source": {
#             "type": "base64",
#             "media_type": "image/png",
#             "data": img_bytes,
#         },
#     },
#     {
#         "type": "text",
#         "text": prompt,
#     },
# ])

# thinking_response = chat(
#     messages,
#     temperature=1.0,
#     thinking=True,
#     thinking_budget=2048,
# )

# print(text_from_message(thinking_response))


In [17]:
# thinking_response.content[0]

In [23]:
# Example: use extended thinking with the same image prompt

with open("earth.pdf", "rb") as f:
    pdf_bytes = base64.b64encode(f.read()).decode("utf-8")

messages = []

add_user_message(messages, [
   {
    "type": "document",
    "source": {
        "type": "base64",
        "media_type": "application/pdf",
        "data": pdf_bytes
    }
},
    {
        "type": "text",
        "text": "summarise this decument in detail with good formating",
    },
])

thinking_response = chat(
    messages,
    temperature=1.0,
    thinking=True,
    thinking_budget=2048,
)

print(text_from_message(thinking_response))


# EARTH - Detailed Summary

## Overview

**Earth** is the third planet from the Sun and the only known astronomical object to harbor life. Key defining features:

- **Ocean World**: The only planet in the Solar System with liquid surface water
- **Water Coverage**: 70.8% ocean, 29.2% land
- **Life Support**: Dynamic atmosphere protects surface and sustains life
- **Magnetic Protection**: Liquid outer core generates magnetosphere that deflects solar winds and cosmic radiation

---

## Physical Characteristics

### Size and Structure
- **Mean Radius**: 6,371.0 km
- **Equatorial Radius**: 6,378.137 km
- **Polar Radius**: 6,356.752 km
- **Circumference**: ~40,000 km (25,000 miles)
- **Shape**: Ellipsoid (slightly flattened at poles)
- **Surface Area**: 510,072,000 km²
  - Land: 148,940,000 km²
  - Water: 361,132,000 km²

### Mass and Density
- **Mass**: 5.972 × 10²⁴ kg
- **Density**: 5.513 g/cm³ (densest planet in Solar System)
- **Surface Gravity**: 9.807 m/s² (1 g₀)
- **Escape Velocity**

In [27]:
print(text_from_message(thinking_response).replace('###','').replace('**','').replace('##','').replace("---","").replace("...",""))

# EARTH - Detailed Summary

 Overview

Earth is the third planet from the Sun and the only known astronomical object to harbor life. Key defining features:

- Ocean World: The only planet in the Solar System with liquid surface water
- Water Coverage: 70.8% ocean, 29.2% land
- Life Support: Dynamic atmosphere protects surface and sustains life
- Magnetic Protection: Liquid outer core generates magnetosphere that deflects solar winds and cosmic radiation



 Physical Characteristics

 Size and Structure
- Mean Radius: 6,371.0 km
- Equatorial Radius: 6,378.137 km
- Polar Radius: 6,356.752 km
- Circumference: ~40,000 km (25,000 miles)
- Shape: Ellipsoid (slightly flattened at poles)
- Surface Area: 510,072,000 km²
  - Land: 148,940,000 km²
  - Water: 361,132,000 km²

 Mass and Density
- Mass: 5.972 × 10²⁴ kg
- Density: 5.513 g/cm³ (densest planet in Solar System)
- Surface Gravity: 9.807 m/s² (1 g₀)
- Escape Velocity: 11.186 km/s

 Orbital Characteristics
- Distance from Sun: ~149.6 milli